In [0]:
#Display Databricks Datasets
display(dbutils.fs.ls("/databricks-datasets"))

In [0]:
#Explore the content of credit card fraud dataset
display(dbutils.fs.ls("/databricks-datasets/credit-card-fraud/"))


In [0]:
# Dsiplay the description.txt file
display(dbutils.fs.head("/databricks-datasets/credit-card-fraud/description.txt"))

In [0]:
#Path to credit card fraud dataset data
path = "/databricks-datasets/credit-card-fraud/data/"

# Load data into df_bronze dataframe
# Use inferSchema option to infer schema
# Use header option to indicate that first line of data is header
df_bronze = spark.read.format("parquet").option("header", "true").option("inferSchema", "true").load(path)

# Verify load and schema (data type)
# Count number of records
# Display 10 records
df_bronze.count()
df_bronze.printSchema()
df_bronze.show(10)
#Read data into dataframe

######After reviewing the `description.txt` file, I discovered that only 0.172% of the transactions in this dataset are labeled as fraud. This highlights the significant class imbalance and the importance of accurately identifying fraudulent transactions. My next step is to design a data pipeline that can effectively track and analyze these rare fraud cases throughout the data processing workflow.

###### From `description.txt` it says that column amountRange is a code from 0 to 7, I will use funtion when to create a new column with the amountrange labeled.

In [0]:
# Create the df_silver
# Create new column based on amountRange to label the amount range

from pyspark.sql.functions import col, when

df_silver = df_bronze.withColumn("amount_bracket", when
    (col("amountRange") == 0, "0-1 USD")
    .when(col("amountRange") == 1, "1-5 USD")
    .when(col("amountRange") == 2, "5-10 USD")
    .when(col("amountRange") == 3, "10-20 USD")
    .when(col("amountRange") == 4, "20-50 USD")
    .when(col("amountRange") == 5, "50-100 USD")
    .when(col("amountRange") == 6, "100-200 USD")
    .when(col("amountRange") == 7, "200+ USD")
    .otherwise("Unknown")
)

# Selecting and reordering columns for the final Silver table
# 'label' 1 = Fraud, 0 = Legitimate
df_silver_final = df_silver.select(
    col("time").alias("seconds_elapsed"),
    col("amount_bracket"),
    col("label").alias("is_fraud"),
    "pcaVector"
)

display(df_silver_final.show(10))

I want to do the `df_gold` layer in sql so I will create a view in python, therefore I can use it in sql as a table

In [0]:
 #Create the df as a SQL temporary view
 df_silver_final.createOrReplaceTempView("silver_fraud")

In [0]:
%sql
--Gold layer: analyzing fraud per bracket 
SELECT 
    amount_bracket,
    COUNT (*) AS total_transactions,
    ROUND(AVG(is_fraud) * 100, 2) AS fraud_rate_percentage
FROM
    silver_fraud
GROUP BY
    amount_bracket
ORDER BY
    fraud_rate_percentage DESC

--Here are the DataFrames in the session:
--- Spark DataFrame 'df_bronze', columns = ['time': integer, 'amountRange': integer, 'label': integer, 'pcaVector': udt]

Databricks visualization. Run in Databricks to view.

I want to know if there is any correlation between the `total_transactions` and `fraud_rate_percentage`

In [0]:
%sql
--Create a view using _sqldf
WITH bracket_metrics AS (
SELECT 
    amount_bracket,
    COUNT(*) AS total_transactions,
    AVG(is_fraud) * 100 AS fraud_rate_percentage
  FROM silver_fraud
  GROUP BY amount_bracket
)
SELECT CORR(total_transactions, fraud_rate_percentage) AS pearson_corr FROM bracket_metrics

Exists a negative moderate correlation between volume and risk